### This notebook creates Gold-layer features from cleaned Silver data for analytics and machine learning.

 Gold Layer – AI-Oriented Feature Design:

In the Gold layer, cleaned Silver data will be transformed into AI-ready features designed to help models learn COVID risk patterns.  
Rather than using raw counts, features such as growth rates, ratios, and vaccination coverage are planned to capture trends, severity, and protection levels. 
 
These features are intended to form the foundation for training interpretable machine learning models that generate actionable public health insights.


### Load Silver tables from Unity Catalog

In [0]:
covid_silver_df = spark.read.table("workspace.default.covid_silver")
vaccination_silver_df = spark.read.table("workspace.default.vaccination_silver")


### Verify data is loaded

In [0]:
display(covid_silver_df.limit(5))


date,new_cases,cum_cases,new_death,cum_death,new_recovered,cum_recovered,cum_active_cases
2020-01-30,1,1,0,0,0,0,1
2020-01-31,0,1,0,0,0,0,1
2020-02-01,0,1,0,0,0,0,1
2020-02-02,1,2,0,0,0,0,2
2020-02-03,1,3,0,0,0,0,3


In [0]:
display(vaccination_silver_df.limit(5))


date,total_vaccinations,people_vaccinated,people_fully_vaccinated,daily_vaccinations_raw,daily_vaccinations,total_vaccinations_per_hundred,people_vaccinated_per_hundred
2021-01-15,0,0,null,0,0,0.0,0.0
2021-01-16,191181,191181,null,191181,191181,0.01,0.01
2021-01-17,224301,224301,null,33120,112150,0.02,0.02
2021-01-18,454049,454049,null,229748,151350,0.03,0.03
2021-01-19,674835,674835,null,220786,168709,0.05,0.05


### Create First Gold Feature
A daily case growth rate feature is created to capture COVID spread trends using day-over-day case changes.


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag, try_divide

# Window ordered by date
window_spec = Window.orderBy("date")

covid_gold_df = (
    covid_silver_df
    .withColumn("previous_day_cases", lag(col("new_cases")).over(window_spec))
    .withColumn(
        "daily_case_growth_rate",
        try_divide(
            col("new_cases") - col("previous_day_cases"),
            col("previous_day_cases")
        )
    )
)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(
    covid_gold_df.select("date", "new_cases", "daily_case_growth_rate").limit(10)
)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


date,new_cases,daily_case_growth_rate
2020-01-30,1,null
2020-01-31,0,-1.0
2020-02-01,0,null
2020-02-02,1,null
2020-02-03,1,0.0
2020-02-04,0,-1.0
2020-02-05,0,null
2020-02-06,0,null
2020-02-07,0,null
2020-02-08,0,null


### Create Active Case Ratio

In [0]:
covid_gold_df.printSchema()


root
 |-- date: date (nullable = true)
 |-- new_cases: integer (nullable = true)
 |-- cum_cases: integer (nullable = true)
 |-- new_death: integer (nullable = true)
 |-- cum_death: integer (nullable = true)
 |-- new_recovered: integer (nullable = true)
 |-- cum_recovered: integer (nullable = true)
 |-- cum_active_cases: integer (nullable = true)
 |-- previous_day_cases: integer (nullable = true)
 |-- daily_case_growth_rate: double (nullable = true)



In [0]:
from pyspark.sql.functions import col, try_divide

covid_gold_df = (
    covid_gold_df
    .withColumn(
        "active_case_ratio",
        try_divide(col("cum_active_cases"), col("cum_cases"))
    )
)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(
    covid_gold_df.select(
        "date",
        "cum_cases",
        "cum_active_cases",
        "active_case_ratio"
    ).limit(30)
)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


date,cum_cases,cum_active_cases,active_case_ratio
2020-01-30,1,1,1.0
2020-01-31,1,1,1.0
2020-02-01,1,1,1.0
2020-02-02,2,2,1.0
2020-02-03,3,3,1.0
2020-02-04,3,3,1.0
2020-02-05,3,3,1.0
2020-02-06,3,3,1.0
2020-02-07,3,3,1.0
2020-02-08,3,3,1.0


### Create Mortality Rate

In [0]:
from pyspark.sql.functions import col, try_divide

covid_gold_df = (
    covid_gold_df
    .withColumn(
        "mortality_rate",
        try_divide(col("cum_death"), col("cum_cases"))
    )
)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(
    covid_gold_df.select(
        "date",
        "cum_cases",
        "cum_death",
        "mortality_rate"
    ).limit(10)
)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


date,cum_cases,cum_death,mortality_rate
2020-01-30,1,0,0.0
2020-01-31,1,0,0.0
2020-02-01,1,0,0.0
2020-02-02,2,0,0.0
2020-02-03,3,0,0.0
2020-02-04,3,0,0.0
2020-02-05,3,0,0.0
2020-02-06,3,0,0.0
2020-02-07,3,0,0.0
2020-02-08,3,0,0.0


### Save Gold Features as Delta Tables

In [0]:
covid_gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.covid_gold")


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
#Verify Gold table exists
spark.sql("SHOW TABLES IN workspace.default").show()


+--------+------------------+-----------+
|database|         tableName|isTemporary|
+--------+------------------+-----------+
| default|        covid_gold|      false|
| default|      covid_silver|      false|
| default|vaccination_silver|      false|
+--------+------------------+-----------+

